# **Data Lake Merge Assignment**

Importing Required Libraries

In [0]:
from delta.tables import DeltaTable

Loading CSV File / Loading Dataset 

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Workspace/Assignment_7/superstore.csv")

display(df.limit(20))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


Renaming column names to make them compatible with Delta Lake.

In [0]:
for col in df.columns:
    df = df.withColumnRenamed(col, col.replace(" ", "_"))

Exploring the Data 

In [0]:
print("Number of Rows:", df.count())

Number of Rows: 9994


In [0]:
print("Number of Columns :", len(df.columns))

Number of Columns : 21


In [0]:
df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



Cleaing Dataset

In [0]:
clean_df = df.dropna()

In [0]:
clean_df = clean_df.dropDuplicates()

In [0]:
print("Rows after cleaning :", clean_df.count())
display(clean_df.limit(20))

Rows after cleaning : 9994


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0,20.6976
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0,218.2518
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0,3.4368
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203


In [0]:
clean_df = clean_df.dropDuplicates(["Order_ID", "Product_ID"])

In [0]:
clean_df.groupBy("Order_ID", "Product_ID").count().filter("count > 1").show()

+--------+----------+-----+
|Order_ID|Product_ID|count|
+--------+----------+-----+
+--------+----------+-----+



Created the Delta Table

In [0]:
table_name = "superstore_delta"

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

Loading The Incremented Dataset

In [0]:
incremental_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Workspace/Assignment_7/superstore_incremental.csv")

display(incremental_df.limit(20))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
5685,CA-2014-164315,2014-04-02,2014-04-08,Standard Class,RD-19585,Rob Dowd,Consumer,United States,Athens,Georgia,30605,South,OFF-AP-10003842,Office Supplies,Appliances,Euro-Pro Shark Turbo Vacuum,185.88,5,0.0,44.3
8441,CA-2016-158302,2016-12-08,2016-12-12,Second Class,JB-16045,Julia Barnett,Home Office,United States,Columbus,Ohio,43229,East,OFF-PA-10002615,Office Supplies,Paper,"""Ampad Gold Fibre Wirebound Steno Books, 6"""" x 9""""","Gregg Ruled""",12.7,3,0.2
857,CA-2014-125612,2014-08-03,2014-08-08,Standard Class,BK-11260,Berenike Kampe,Consumer,United States,New York City,New York,10035,East,OFF-SU-10002537,Office Supplies,Supplies,Acme Box Cutter Scissors,122.76,10,0.0,29.26
4066,CA-2014-102673,2014-11-01,2014-11-05,Standard Class,KH-16630,Ken Heidel,Corporate,United States,Charlotte,North Carolina,28205,South,OFF-FA-10003021,Office Supplies,Fasteners,Staples,9.02,5,0.2,1.55
4309,NEW-10008,2014-11-04,2014-11-11,Standard Class,WB-21850,William Brown,Consumer,United States,Los Angeles,California,90045,West,TEC-PH-10001079,Technology,Phones,Polycom SoundPoint Pro SE-225 Corded phone,666.344,7,0.2,66.6344
9353,CA-2017-148411,2017-09-24,2017-09-26,First Class,RO-19780,Rose O'Brian,Consumer,United States,Chicago,Illinois,60623,Central,FUR-CH-10003973,Furniture,Chairs,GuestStacker Chair with Chrome Finish Legs,624.56,2,0.3,-16.36
32,US-2015-150630,2015-09-17,2015-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,OFF-AR-10004042,Office Supplies,Art,"BOSTON Model 1800 Electric Pencil Sharpeners, Putty/Woodgrain",103.56,6,0.2,10.68
6939,NEW-10005,2016-11-05,2016-11-09,Standard Class,NP-18700,Nora Preis,Consumer,United States,Seattle,Washington,98105,West,TEC-AC-10004420,Technology,Accessories,Cherry 142-key Programmable Keyboard,479.72,4,0.0,52.7692
6490,NEW-10016,2017-09-19,2017-09-24,Standard Class,JM-15535,Jessica Myrick,Consumer,United States,New York City,New York,10011,East,OFF-PA-10004782,Office Supplies,Paper,Xerox 228,32.4,5,0.0,15.552
5263,CA-2014-105165,2014-09-07,2014-09-10,First Class,SZ-20035,Sam Zeldin,Home Office,United States,Houston,Texas,77036,Central,FUR-TA-10004154,Furniture,Tables,"Riverside Furniture Oval Coffee Table, Oval End Table, End Table with Drawer",240.95,1,0.3,-25.24


Renaming column names to make them compatible with Delta Lake.

In [0]:
for col in incremental_df.columns:
    incremental_df = incremental_df.withColumnRenamed(col, col.replace(" ", "_"))

Exploring the Incremented Dataset 

In [0]:
print("Number of Rows:", incremental_df.count())    

Number of Rows: 250


In [0]:
print("Number of Columns :", len(incremental_df.columns))

Number of Columns : 21


Cleaning The Incremented Dataset 

In [0]:
incremental_df = incremental_df.dropna()

In [0]:
incremental_df= incremental_df.dropDuplicates()

In [0]:
incremental_df = incremental_df.dropDuplicates(["Order_ID", "Product_ID"])

In [0]:
print("Rows after cleaning :", incremental_df.count())


Rows after cleaning : 250


Created The Incremented Delta Table

Created Incremental Delta Table

In [0]:
table_name2 = "superstore_incremental"
incremental_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(table_name2)

In [0]:
main_table = DeltaTable.forName(spark,"superstore_delta")
incremental_table = DeltaTable.forName(spark,"superstore_incremental")

Merge The Merge Opertations 

In [0]:
merge_result = main_table.alias("target") \
.merge(
    incremental_table.toDF().alias("source"),
    """
    target.Order_ID = source.Order_ID
    AND
    target.Product_ID = source.Product_ID
    """
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
merge_result.show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|              250|             200|               0|               50|
+-----------------+----------------+----------------+-----------------+



Displaying The Final Delta Table

In [0]:
final_df = spark.table("superstore_delta")
display(final_df.limit(20))   

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
4309,NEW-10008,2014-11-04,2014-11-11,Standard Class,WB-21850,William Brown,Consumer,United States,Los Angeles,California,90045,West,TEC-PH-10001079,Technology,Phones,Polycom SoundPoint Pro SE-225 Corded phone,666.344,7,0.2,66.6344
6923,NEW-10045,2015-03-16,2015-03-19,Second Class,MS-17710,Maurice Satty,Consumer,United States,Auburn,Alabama,36830,South,OFF-LA-10003930,Office Supplies,Labels,"Dot Matrix Printer Tape Reel Labels, White, 5000/Box",491.55,5,0.0,240.8595
2996,US-2014-150532,2014-07-14,2014-07-21,Standard Class,PB-19150,Philip Brown,Consumer,United States,Phoenix,Arizona,85023,West,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,67.1,5,0.2,6.92
1761,CA-2015-130785,2015-09-05,2015-09-09,Standard Class,AG-10900,Arthur Gainer,Consumer,United States,San Diego,California,92105,West,FUR-BO-10000330,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Planked Cherry Finish",493.6,4,0.15,-5.32
7475,CA-2014-167199,2014-01-06,2014-01-10,Standard Class,ME-17320,Maria Etezadi,Home Office,United States,Henderson,Kentucky,42420,South,FUR-CH-10004063,Furniture,Chairs,Global Deluxe High-Back Manager's Chair,3088.58,9,0.0,821.05
1727,CA-2015-127453,2015-12-19,2015-12-20,First Class,JK-15370,Jay Kimmel,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AP-10003860,Office Supplies,Appliances,Fellowes Advanced 8 Outlet Surge Suppressor with Phone/Fax Protection,106.6,4,0.2,8.55
3119,CA-2015-121720,2015-06-11,2015-06-12,First Class,JE-15610,Jim Epp,Corporate,United States,Lakeland,Florida,33801,South,FUR-CH-10003312,Furniture,Chairs,Hon 2090 Pillow Soft Series Mid Back Swivel/Tilt Chairs,1348.7,5,0.2,-200.9
6221,CA-2016-160220,2016-10-20,2016-10-26,Standard Class,JS-16030,Joy Smith,Consumer,United States,Trenton,Michigan,48183,Central,OFF-ST-10000617,Office Supplies,Storage,Woodgrain Magazine Files by Perma,25.03,7,0.0,1.61
5291,NEW-10001,2014-09-08,2014-09-15,Standard Class,KT-16465,Kean Takahito,Consumer,United States,Houston,Texas,77036,Central,FUR-CH-10004287,Furniture,Chairs,SAFCO Arco Folding Chair,966.7,5,0.3,-13.81
877,NEW-10032,2016-01-17,2016-01-21,Standard Class,CC-12430,Chuck Clark,Home Office,United States,Concord,New Hampshire,3301,East,TEC-AC-10003628,Technology,Accessories,Logitech 910-002974 M325 Wireless Mouse for Web Scrolling,29.99,1,0.0,13.1956


Validate the Results

In [0]:
print("Total Records after MERGE:", final_df.count())

Total Records after MERGE: 10036


In [0]:
duplicates = final_df.groupBy("Order_ID", "Product_ID").count().filter("count > 1")

display(duplicates)

Order_ID,Product_ID,count
